# 第 15 章　法模態：廣義特徵值問題 $B\mathbf x=\omega^2A\mathbf x$

> 對應 Riley 第 9 章（9.1 典型振盪系統、9.2 對稱性與法模態、9.3 Rayleigh–Ritz 法），
> 並銜接 Strang 6.3（正定）、6.5（微分方程）與第 22 章（變分法）。

小振盪的物理：動能與位能都是**二次式**

$$T=\tfrac12\dot{\mathbf q}^{\mathsf T}A\dot{\mathbf q}\ (\text{質量矩陣}),\qquad
V=\tfrac12\mathbf q^{\mathsf T}B\mathbf q\ (\text{剛度矩陣})$$

能量守恆 $\frac{d}{dt}(T+V)=0$ 給出 $A\ddot{\mathbf q}+B\mathbf q=\mathbf 0$。
代入 $\mathbf q=\mathbf x\cos\omega t$：

$$\boxed{(B-\omega^2A)\mathbf x=\mathbf 0}
\qquad\Longleftrightarrow\qquad \det(B-\omega^2A)=0$$

這是**廣義特徵值問題**。關鍵差異：特徵向量不是在普通內積下正交，而是

$$\mathbf x_i^{\mathsf T}A\mathbf x_j=0,\qquad \mathbf x_i^{\mathsf T}B\mathbf x_j=0
\quad(i\ne j)\qquad\text{但 }\mathbf x_i^{\mathsf T}\mathbf x_j\ne0$$

```bash
python chapters/ch15_normal_modes.py
python tools/build_notebooks.py ch15
```

In [ ]:
import pathlib
import sys

_r = pathlib.Path(globals().get("__file__", "_")).resolve().parent.parent
if not (_r / "linalg_tutorial").is_dir():
    _r = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "linalg_tutorial").is_dir())
sys.path.insert(0, str(_r))

import numpy as np

from linalg_tutorial.eigen import generalized_symmetric_eig, is_positive_definite
from linalg_tutorial.elimination import cholesky, second_difference_matrix
from linalg_tutorial.utils import check, section, show_matrix
from linalg_tutorial.viz import finish, new_axes, plt

np.set_printoptions(precision=4, suppress=True)

## 15.1　為什麼是廣義特徵值問題

若質量矩陣 $A=I$（所有質量相同且座標已正規化），問題退化成普通的
$B\mathbf x=\omega^2\mathbf x$。一般情況下 $A\ne I$，要先「把 $A$ 吸收掉」：

$$A=R^{\mathsf T}R\ (\text{Cholesky})\quad\Longrightarrow\quad
\underbrace{R^{-\mathsf T}BR^{-1}}_{\text{對稱！}}\mathbf y=\omega^2\mathbf y,
\qquad \mathbf x=R^{-1}\mathbf y$$

因為 $R^{-\mathsf T}BR^{-1}$ 仍然對稱，所以

* $\omega^2$ 全為**實數**（$A,B$ 正定 $\Rightarrow\omega^2>0$）
* 特徵向量在 **$A$-內積** $\langle\mathbf x,\mathbf y\rangle_A=\mathbf x^{\mathsf T}A\mathbf y$ 下正交

In [ ]:
section("15.1 Riley 的例子：細桿 + 輕繩的雙擺")

# T = (Ml²/12) q̇ᵀ[[6,3],[3,2]] q̇ / ... ⇒ A, B（公因子不影響 λ = ω²l/g）
A = np.array([[6.0, 3.0], [3.0, 2.0]])          # 質量矩陣（乘 Ml²/12）
B = np.array([[6.0, 0.0], [0.0, 3.0]])          # 剛度矩陣（乘 Mlg/12）
show_matrix("質量矩陣 A", A)
show_matrix("剛度矩陣 B", B)
check("A 對稱正定（動能恆正）", is_positive_definite(A))
check("B 對稱正定（位能極小）", is_positive_definite(B))

# det(B − λA) = 0 ⇒ λ² − 10λ + 6 = 0
lam_theory = np.array([5 - np.sqrt(19), 5 + np.sqrt(19)])
print(f"\n特徵方程 |B − λA| = 0 ⇒ λ² − 10λ + 6 = 0 ⇒ λ = 5 ± √19 = {lam_theory}")
lam, X = generalized_symmetric_eig(B, A)
print(f"數值解 λ = ω²l/g = {lam}")
check("λ = 5 ± √19", np.sort(lam), np.sort(lam_theory))
print(f"法頻率 ω₁ = √({lam[0]:.4f} g/l)，ω₂ = √({lam[1]:.4f} g/l)")
check("ω₁² ≈ 0.641 g/l", lam[0], 0.6412, tol=1e-3)
check("ω₂² ≈ 9.359 g/l", lam[1], 9.3588, tol=1e-3)

# 驗證廣義特徵值方程與 A-正交性
for k in range(2):
    check(f"模態 {k + 1}：Bx = λAx", B @ X[:, k], lam[k] * (A @ X[:, k]))
print(f"\n模態 1（低頻）：x = {np.round(X[:, 0] / X[0, 0], 4)}"
      f"  → 繩與桿幾乎成一直線（像單擺）")
print(f"模態 2（高頻）：x = {np.round(X[:, 1] / X[1 - 1, 1], 4)}"
      f"  → 繩與桿反向擺動")
check("模態 1 的兩分量同號（同向）", X[0, 0] * X[1, 0] > 0)
check("模態 2 的兩分量異號（反向）", X[0, 1] * X[1, 1] < 0)

print("\nA-正交性（物理上的正交，不是幾何正交）：")
print(f"  x₁ᵀAx₂ = {X[:, 0] @ A @ X[:, 1]:.3e}")
print(f"  x₁ᵀBx₂ = {X[:, 0] @ B @ X[:, 1]:.3e}")
print(f"  x₁ᵀx₂  = {X[:, 0] @ X[:, 1]:.6f}  ← 一般不為 0！")
check("x₁ᵀAx₂ = 0", X[:, 0] @ A @ X[:, 1], 0.0)
check("x₁ᵀBx₂ = 0", X[:, 0] @ B @ X[:, 1], 0.0)
check("x₁ᵀx₂ ≠ 0（普通內積下不正交）", abs(X[:, 0] @ X[:, 1]) > 1e-6)
check("歸一化：xᵏᵀAxᵏ = 1", np.diag(X.T @ A @ X), np.ones(2))
check("XᵀAX = I 且 XᵀBX = diag(λ)（同時對角化！）", X.T @ B @ X, np.diag(lam))

# 與 scipy 的廣義特徵值求解器比較
try:
    from scipy.linalg import eigh
    lam_sp, X_sp = eigh(B, A)
    check("與 scipy.linalg.eigh(B, A) 一致", np.sort(lam_sp), np.sort(lam))
except ImportError:
    print("  （未安裝 scipy，略過比較）")

### 法座標：把耦合的方程解耦

令 $\mathbf q=X\boldsymbol\eta$（$X$ 的欄是法模態），則

$$X^{\mathsf T}AX=I,\quad X^{\mathsf T}BX=\operatorname{diag}(\omega_k^2)
\quad\Longrightarrow\quad \ddot\eta_k+\omega_k^2\eta_k=0$$

**$N$ 個耦合的振盪子變成 $N$ 個獨立的單擺**。
任意運動都是法模態的疊加 ── 這正是第 17 章 Fourier 分析的有限維版本。

In [ ]:
section("15.1 法座標讓方程完全解耦")

q0 = np.array([0.1, 0.0])                       # 初始位移
qdot0 = np.array([0.0, 0.0])                    # 初速為 0
eta0 = np.linalg.solve(X, q0)
print(f"初始條件 q(0) = {q0} 的法座標 η(0) = {np.round(eta0, 6)}")


def q_exact(t):
    """q(t) = Σ ηₖ(0) cos(ωₖt) xₖ（每個法模態各自簡諧振盪）。"""
    return X @ (eta0 * np.cos(np.sqrt(lam) * t))


# 直接數值積分 A q̈ + B q = 0 做對照
def simulate(t_end, dt=1e-4):
    """用 leapfrog 積分 Aq̈ = −Bq。"""
    q, v = q0.copy(), qdot0.copy()
    Ainv = np.linalg.inv(A)
    steps = int(t_end / dt)
    for _ in range(steps):
        a = -Ainv @ (B @ q)
        v = v + dt * a
        q = q + dt * v
    return q


print(f"\n{'t':>6} | {'法模態疊加':<26} | {'數值積分':<26} | 誤差")
print("-" * 76)
for t in [0.5, 1.0, 2.0]:
    qa, qn = q_exact(t), simulate(t)
    print(f"{t:>6} | {str(np.round(qa, 6)):<26} | {str(np.round(qn, 6)):<26} "
          f"| {np.linalg.norm(qa - qn):.2e}")
    check(f"  t = {t}：兩者相符", qa, qn, tol=5e-4)
check("能量守恆（法模態解）",
      0.5 * (X @ (-eta0 * np.sqrt(lam) * np.sin(np.sqrt(lam) * 1.3))) @ A @
      (X @ (-eta0 * np.sqrt(lam) * np.sin(np.sqrt(lam) * 1.3)))
      + 0.5 * q_exact(1.3) @ B @ q_exact(1.3),
      0.5 * q0 @ B @ q0, tol=1e-9)

## 15.2　線性三原子分子（CO₂ 模型）

三個質點 $m,\mu m,m$ 用兩根相同的彈簧（力常數 $k$）串成一直線：

$$T=\tfrac12m(\dot x_1^2+\mu\dot x_2^2+\dot x_3^2),\qquad
V=\tfrac12k\left[(x_2-x_1)^2+(x_3-x_2)^2\right]$$

三個法模態（理論值）：

| 模態 | $\omega^2$ | 位移形式 | 物理 |
|---|---|---|---|
| 1 | $0$ | $(1,1,1)$ | **整體平移**（零模態）|
| 2 | $k/m$ | $(1,0,-1)$ | 兩端反向，中間不動 |
| 3 | $\dfrac{k}{m}\left(1+\dfrac{2}{\mu}\right)$ | $(1,-2/\mu,1)$ | 中間與兩端反向 |

$\omega=0$ 的零模態對應 $B$ 的零空間（$B\mathbf 1=\mathbf 0$）──
正是第 2 章那個「自由–自由」奇異矩陣！

In [ ]:
section("15.2 CO₂ 的三個法模態")

m, mu, k = 1.0, 16.0 / 12.0, 1.0                # CO₂：碳 12、氧 16 ⇒ μ = 12/16
mu = 12.0 / 16.0                                 # 中央碳原子相對氧原子的質量比
A_co2 = m * np.diag([1.0, mu, 1.0])
B_co2 = k * np.array([[1.0, -1.0, 0.0], [-1.0, 2.0, -1.0], [0.0, -1.0, 1.0]])
show_matrix("質量矩陣 A", A_co2)
show_matrix("剛度矩陣 B（= 第 2 章的自由–自由矩陣！）", B_co2)
check("B 奇異（有零模態）", abs(np.linalg.det(B_co2)) < 1e-12)
check("B·(1,1,1) = 0（整體平移不花位能）", B_co2 @ np.ones(3), np.zeros(3))
check("B 半正定（位能 ≥ 0）", np.all(np.linalg.eigvalsh(B_co2) > -1e-12))

# A 正定所以可以用 Cholesky 轉成標準問題
R = cholesky(A_co2)
S_std = np.linalg.inv(R).T @ B_co2 @ np.linalg.inv(R)
check("R⁻ᵀBR⁻¹ 對稱 ⇒ ω² 必為實數", S_std, S_std.T)
omega2, modes = generalized_symmetric_eig(B_co2, A_co2)
print(f"\nω² = {omega2}")
theory = np.array([0.0, k / m, k / m * (1 + 2 / mu)])
print(f"理論 = {np.sort(theory)}")
check("ω² 與理論值相符", np.sort(omega2), np.sort(theory))

names = ["平移（ω = 0）", "反對稱伸縮", "對稱伸縮"]
for idx, name in zip(np.argsort(omega2), names):
    v = modes[:, idx] / modes[np.argmax(np.abs(modes[:, idx])), idx]
    print(f"  {name:<16} ω² = {omega2[idx]:>8.4f}，位移比 = {np.round(v, 4)}")
check("零模態 = (1,1,1)（整體平移）",
      np.abs(modes[:, np.argmin(omega2)] / modes[0, np.argmin(omega2)]),
      np.ones(3), tol=1e-8)
idx_mid = np.argsort(omega2)[1]
v_mid = modes[:, idx_mid] / modes[0, idx_mid]
check("第二模態 = (1,0,−1)（中間原子不動）", v_mid, np.array([1.0, 0.0, -1.0]),
      tol=1e-8)
idx_hi = np.argsort(omega2)[2]
v_hi = modes[:, idx_hi] / modes[0, idx_hi]
check("第三模態 = (1,−2/μ,1)", v_hi, np.array([1.0, -2 / mu, 1.0]), tol=1e-8)

print("\n動量守恆的檢查（質量加權的位移總和）：")
for idx, name in zip(np.argsort(omega2), names):
    p_total = A_co2 @ modes[:, idx]
    print(f"  {name:<16} Σmᵢxᵢ = {p_total.sum():>10.3e}")
    if omega2[idx] > 1e-10:
        check(f"  {name}：振動模態的質心不動（Σmᵢxᵢ = 0）", p_total.sum(), 0.0)
print("  ⇒ 振動模態保持質心靜止；只有零模態在移動質心（那就是平移）")
check("模態之間 A-正交", modes.T @ A_co2 @ modes, np.eye(3), tol=1e-10)
check("XᵀBX = diag(ω²)", modes.T @ B_co2 @ modes, np.diag(omega2), tol=1e-10)

fig, ax = new_axes("", figsize=(8.0, 3.4), equal=False)
fig.clf()
for p, (idx, name) in enumerate(zip(np.argsort(omega2), names)):
    axk = fig.add_subplot(1, 3, p + 1)
    v = modes[:, idx] / np.max(np.abs(modes[:, idx]))
    base = np.array([0.0, 1.0, 2.0])
    axk.plot(base, [0, 0, 0], "ko", ms=9)
    for xb, dv in zip(base, v):
        axk.annotate("", xy=(xb + 0.35 * dv, 0), xytext=(xb, 0),
                     arrowprops=dict(arrowstyle="-|>", color="C3", lw=2.2))
    axk.set_title(f"{['translation', 'antisymmetric', 'symmetric'][p]}\n"
                  f"omega^2 = {omega2[idx]:.3f}", fontsize=9)
    axk.set_xlim(-0.8, 2.8)
    axk.set_ylim(-0.6, 0.6)
    axk.set_yticks([])
    axk.grid(alpha=0.2)
finish(fig, "ch15_co2_modes")

## 15.3　$N$ 個質點的彈簧鏈 → 連續極限

$N$ 個質量 $m$、$N+1$ 根彈簧（兩端固定）：
$A=mI$、$B=k\,\text{tridiag}(-1,2,-1)=kK$ ── **又是第 2 章的 $K$**！

於是法模態就是**離散正弦波**（第 7 章已證明）：

$$x_j^{(n)}=\sin\frac{jn\pi}{N+1},\qquad
\omega_n^2=\frac{4k}{m}\sin^2\frac{n\pi}{2(N+1)}$$

低頻（$n\ll N$）時 $\omega_n\approx\frac{n\pi}{N+1}\sqrt{k/m}$ ──
正是**弦的振動**（連續極限 $\omega_n\propto n$，泛音列！）。

In [ ]:
section("15.3 彈簧鏈的法模態 = 離散正弦波")

N = 8
m_s, k_s = 1.0, 1.0
A_chain = m_s * np.eye(N)
B_chain = k_s * second_difference_matrix(N)
omega2_c, modes_c = generalized_symmetric_eig(B_chain, A_chain)
order = np.argsort(omega2_c)
omega2_c, modes_c = omega2_c[order], modes_c[:, order]

print(f"{'n':>3} | {'ω² (數值)':>12} | {'4k/m·sin²(nπ/2(N+1))':>22} | "
      f"{'ωₙ/ω₁':>8} | {'連續極限 n':>10}")
print("-" * 68)
for n in range(1, N + 1):
    theory = 4 * k_s / m_s * np.sin(n * np.pi / (2 * (N + 1))) ** 2
    print(f"{n:>3} | {omega2_c[n - 1]:>12.8f} | {theory:>22.8f} | "
          f"{np.sqrt(omega2_c[n - 1] / omega2_c[0]):>8.4f} | {n:>10}")
    check(f"  n = {n}：ωₙ² 與理論相符", omega2_c[n - 1], theory)

j = np.arange(1, N + 1)
for n in [1, 2, 3]:
    v_theory = np.sin(j * n * np.pi / (N + 1))
    v_theory = v_theory / np.linalg.norm(v_theory)
    v_num = modes_c[:, n - 1] / np.linalg.norm(modes_c[:, n - 1])
    if v_num @ v_theory < 0:
        v_num = -v_num
    check(f"模態 {n} = 離散正弦波 sin(jnπ/(N+1))", v_num, v_theory, tol=1e-8)

print(f"\n低頻模態的頻率比（泛音列）：")
print(f"  ω₁:ω₂:ω₃ = 1 : {np.sqrt(omega2_c[1] / omega2_c[0]):.4f} : "
      f"{np.sqrt(omega2_c[2] / omega2_c[0]):.4f}")
print(f"  連續弦的理論 = 1 : 2 : 3（N 越大越接近）")
for N_big in [8, 50, 400]:
    o2 = np.linalg.eigvalsh(second_difference_matrix(N_big))
    ratio = np.sqrt(o2[1] / o2[0])
    print(f"  N = {N_big:>4}：ω₂/ω₁ = {ratio:.6f}")
check("N 增大時 ω₂/ω₁ → 2（連續極限）",
      np.sqrt(np.linalg.eigvalsh(second_difference_matrix(400))[1]
              / np.linalg.eigvalsh(second_difference_matrix(400))[0]), 2.0, tol=0.01)

fig, ax = new_axes("Normal modes of a mass-spring chain (N = 8)",
                   figsize=(6.4, 4.2), equal=False)
jj = np.linspace(0, N + 1, 300)
for n, c in zip([1, 2, 3, 8], ["C0", "C1", "C2", "C3"]):
    v = np.sin(j * n * np.pi / (N + 1))
    v = v / np.max(np.abs(v))
    ax.plot(j, v + 2.5 * (4 - [1, 2, 3, 8].index(n)), c + "o", ms=6)
    ax.plot(jj, np.sin(jj * n * np.pi / (N + 1)) + 2.5 * (4 - [1, 2, 3, 8].index(n)),
            c, lw=1.2, alpha=0.7, label=f"n = {n}, omega = {np.sqrt(omega2_c[n-1]):.3f}")
    ax.axhline(2.5 * (4 - [1, 2, 3, 8].index(n)), color="k", lw=0.5, alpha=0.3)
ax.set_xlabel("mass index j")
ax.set_yticks([])
ax.legend(fontsize=7, loc="upper right")
finish(fig, "ch15_chain_modes")

## 15.4　對稱性與法模態（分塊對角化）

若系統有對稱性（例如左右鏡像 $P$），則 $P$ 與 $A,B$ **可交換**：

$$PA=AP,\qquad PB=BP\quad\Longrightarrow\quad
\text{可以同時對角化}$$

於是法模態可依**對稱／反對稱**分類，$N\times N$ 的問題分裂成兩個小問題
── 完全不用解大矩陣。這是群表示論（第 23 章）在物理中最直接的應用。

In [ ]:
section("15.4 用對稱性把問題切小")

# CO₂ 的左右鏡像對稱：P 交換 1 和 3
P = np.array([[0.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]])
check("P² = I（鏡像兩次回到原狀）", P @ P, np.eye(3))
check("PA = AP（質量分布左右對稱）", P @ A_co2, A_co2 @ P)
check("PB = BP（彈簧左右對稱）", P @ B_co2, B_co2 @ P)
print("⇒ P 與 A、B 都可交換，可以同時對角化")

# P 的特徵空間：對稱（λ=+1）與反對稱（λ=−1）
lamP, VP = np.linalg.eigh(P)
sym_space = VP[:, lamP > 0]              # λ = +1，維度 2
anti_space = VP[:, lamP < 0]             # λ = −1，維度 1
print(f"\nP 的特徵值 = {lamP}")
print(f"  對稱子空間（λ=+1）維度 = {sym_space.shape[1]}")
print(f"  反對稱子空間（λ=−1）維度 = {anti_space.shape[1]}")

# 在各子空間裡解小問題
for name, basis in [("對稱 (+1)", sym_space), ("反對稱 (−1)", anti_space)]:
    A_sub = basis.T @ A_co2 @ basis
    B_sub = basis.T @ B_co2 @ basis
    # 注意：A_sub⁻¹B_sub 不對稱，要用廣義特徵值解法（或一般的 eigvals）
    o2_sub = (generalized_symmetric_eig(B_sub, A_sub)[0] if A_sub.shape[0] > 1
              else np.array([B_sub[0, 0] / A_sub[0, 0]]))
    print(f"  {name} 子空間：{A_sub.shape[0]}x{A_sub.shape[0]} 的小問題，"
          f"ω² = {np.round(np.real(o2_sub), 6)}")
    for o in np.real(o2_sub):
        check(f"    ω² = {o:.4f} 確實是原問題的解",
              np.min(np.abs(omega2 - o)) < 1e-8)
print("  ⇒ 3x3 的問題被切成 2x2 + 1x1，而且分類有物理意義")

# 模態本身就是 P 的特徵向量
for idx in range(3):
    v = modes[:, idx]
    parity = (P @ v) @ v / (v @ v)
    print(f"  模態 ω² = {omega2[idx]:.4f} 的奇偶性 = {parity:+.4f}"
          f"（{'對稱' if parity > 0 else '反對稱'}）")
    check(f"  模態是 P 的特徵向量（奇偶性 = ±1）", abs(abs(parity) - 1) < 1e-8)

## 15.5　Rayleigh–Ritz：不解方程也能估計頻率

$$\lambda(\mathbf x)=\frac{\mathbf x^{\mathsf T}B\mathbf x}{\mathbf x^{\mathsf T}A\mathbf x}
\qquad\Longrightarrow\qquad
\omega_1^2\le\lambda(\mathbf x)\le\omega_N^2\quad\text{對任何 }\mathbf x$$

任何「試驗向量」都給出 $\omega_1^2$ 的**上界**。
而且因為 $\lambda$ 在特徵向量處**駐定**，試驗向量的誤差是 $O(\epsilon)$ 時，
頻率估計的誤差只有 $O(\epsilon^2)$ ── 這就是 Rayleigh–Ritz 法威力的來源，
也是有限元素法與量子化學變分法的基礎。

In [ ]:
section("15.5 Rayleigh–Ritz：試驗向量給出上界")

N = 10
A_r = np.eye(N)
B_r = second_difference_matrix(N)
omega2_r = np.sort(np.linalg.eigvalsh(B_r))
print(f"真實的 ω₁² = {omega2_r[0]:.8f}，ω_N² = {omega2_r[-1]:.8f}")

rayleigh = lambda x: (x @ B_r @ x) / (x @ A_r @ x)
j = np.arange(1, N + 1)
trials = {
    "常數向量 (1,1,...,1)": np.ones(N),
    "拋物線 j(N+1−j)": j * (N + 1 - j),
    "三角形（尖頂）": np.minimum(j, N + 1 - j).astype(float),
    "正確的正弦波": np.sin(j * np.pi / (N + 1)),
}
print(f"\n{'試驗向量':<24} | {'λ(x)':>12} | {'相對誤差':>12} | {'是上界?':>8}")
print("-" * 64)
for name, x in trials.items():
    val = rayleigh(x)
    err = (val - omega2_r[0]) / omega2_r[0]
    print(f"{name:<24} | {val:>12.8f} | {err:>11.3%} | "
          f"{str(val >= omega2_r[0] - 1e-12):>8}")
    check(f"  {name}：λ(x) ≥ ω₁²（上界性質）", val >= omega2_r[0] - 1e-12)
    check(f"  {name}：λ(x) ≤ ω_N²", val <= omega2_r[-1] + 1e-12)

# 二階精度：試驗向量誤差 ε ⇒ 頻率誤差 ε²
print("\n駐定性 ⇒ 二階精度：")
v1 = np.sin(j * np.pi / (N + 1))
v1 = v1 / np.linalg.norm(v1)
v2 = np.sin(2 * j * np.pi / (N + 1))
v2 = v2 / np.linalg.norm(v2)
print(f"{'擾動 ε':>10} | {'λ 的誤差':>14} | {'誤差/ε²':>12}")
print("-" * 42)
for eps in [0.1, 0.03, 0.01, 0.003]:
    x = v1 + eps * v2
    err = rayleigh(x) - omega2_r[0]
    print(f"{eps:>10} | {err:>14.3e} | {err / eps ** 2:>12.6f}")
check("頻率誤差 ∝ ε²（而非 ε）",
      (rayleigh(v1 + 0.01 * v2) - omega2_r[0]) / 0.01 ** 2,
      (rayleigh(v1 + 0.003 * v2) - omega2_r[0]) / 0.003 ** 2, tol=1e-6)
print("  ⇒ 試驗向量差 10%，頻率只差 1% —— Rayleigh–Ritz 便宜又準")

# Ritz 法：在小子空間裡解廣義特徵值問題
print("\nRitz 法：用 k 個試驗向量張出的子空間求前 k 個頻率")
basis_pool = np.column_stack([j * (N + 1 - j),
                              j * (N + 1 - j) * (2 * j - N - 1),
                              (j * (N + 1 - j)) ** 2,
                              np.minimum(j, N + 1 - j).astype(float)])
print(f"{'k':>3} | {'Ritz 的 ω₁²':>14} | {'真值':>12} | {'Ritz 的 ω₂²':>14} | {'真值':>12}")
print("-" * 64)
for k_ritz in [1, 2, 3, 4]:
    Z = basis_pool[:, :k_ritz]
    A_small = Z.T @ A_r @ Z
    B_small = Z.T @ B_r @ Z
    ritz = np.sort(np.linalg.eigvalsh(np.linalg.solve(cholesky(A_small).T,
                   np.linalg.solve(cholesky(A_small).T, B_small).T).T))
    ritz = np.sort(np.real(np.linalg.eigvals(np.linalg.solve(A_small, B_small))))
    second = f"{ritz[1]:>14.8f}" if len(ritz) > 1 else f"{'—':>14}"
    second_true = f"{omega2_r[1]:>12.8f}" if len(ritz) > 1 else f"{'—':>12}"
    print(f"{k_ritz:>3} | {ritz[0]:>14.8f} | {omega2_r[0]:>12.8f} | {second} | {second_true}")
    check(f"  k = {k_ritz}：Ritz 的 ω₁² ≥ 真值（上界）",
          ritz[0] >= omega2_r[0] - 1e-10)
    if len(ritz) > 1:
        check(f"  k = {k_ritz}：Ritz 的 ω₂² ≥ 真值", ritz[1] >= omega2_r[1] - 1e-10)
print("  ⇒ 子空間越大，估計越準，且永遠從上方逼近（這就是有限元素法）")

## 動手練習

1. 兩個質量 $m$ 用三根彈簧（$k$–$k$–$k$）連在兩面牆之間。求法頻率與模態，
   並用對稱性預測答案。
2. 把 CO₂ 的中央質量 $\mu$ 改成很大或很小，觀察三個頻率怎麼變。
3. 證明：若 $A,B$ 都正定，則所有 $\omega^2>0$（系統穩定）。
4. 用 Rayleigh 商估計 $N=50$ 彈簧鏈的最低頻率，比較常數、拋物線、正弦三種試驗向量。

參考解答：

In [ ]:
section("練習參考解答")

# 練習 1：兩質量三彈簧
A_e = np.eye(2)
B_e = np.array([[2.0, -1.0], [-1.0, 2.0]])
o2_e, X_e = generalized_symmetric_eig(B_e, A_e)
print(f"練習 1：ω² = {np.sort(o2_e)}（理論 k/m 與 3k/m）")
check("  ω² = 1 與 3", np.sort(o2_e), np.array([1.0, 3.0]))
for idx in np.argsort(o2_e):
    v = X_e[:, idx] / X_e[0, idx]
    print(f"  ω² = {o2_e[idx]:.1f}：模態 = {np.round(v, 4)}"
          f"（{'同向（對稱）' if v[1] > 0 else '反向（反對稱）'}）")
P_e = np.array([[0.0, 1.0], [1.0, 0.0]])
check("  P 與 B 可交換 ⇒ 模態必為對稱或反對稱", P_e @ B_e, B_e @ P_e)
check("  對稱模態 (1,1) 的 ω² = 1", B_e @ np.ones(2), 1.0 * np.ones(2))
check("  反對稱模態 (1,−1) 的 ω² = 3",
      B_e @ np.array([1.0, -1.0]), 3.0 * np.array([1.0, -1.0]))

# 練習 2
print("\n練習 2：改變中央質量 μ")
print(f"{'μ':>8} | {'ω₁²':>10} | {'ω₂²':>10} | {'ω₃²':>12} | 說明")
print("-" * 62)
for mu_ in [0.1, 0.75, 1.0, 10.0, 100.0]:
    A_t = np.diag([1.0, mu_, 1.0])
    o2_t = np.sort(generalized_symmetric_eig(B_co2, A_t)[0])
    note = ("中央很輕 ⇒ 高頻極高" if mu_ < 0.3 else
            "中央很重 ⇒ 高頻趨近 k/m" if mu_ > 5 else "")
    print(f"{mu_:>8} | {o2_t[0]:>10.4f} | {o2_t[1]:>10.4f} | {o2_t[2]:>12.4f} | {note}")
    check(f"  μ = {mu_}：ω₂² = k/m 不受 μ 影響", o2_t[1], 1.0)
    check(f"  μ = {mu_}：ω₃² = (1+2/μ)k/m", o2_t[2], 1 + 2 / mu_)
print("  ω₂（兩端反向、中間不動）與中央質量無關 —— 因為中間根本不動！")

# 練習 3
print("\n練習 3：A、B 正定 ⇒ ω² > 0")
print("  證明：ω² = xᵀBx / xᵀAx，分子分母都 > 0（x ≠ 0）⇒ ω² > 0")
rng = np.random.default_rng(0)
for trial in range(3):
    M1 = rng.standard_normal((4, 4))
    A_p = M1 @ M1.T + 4 * np.eye(4)
    M2 = rng.standard_normal((4, 4))
    B_p = M2 @ M2.T + 4 * np.eye(4)
    o2_p = generalized_symmetric_eig(B_p, A_p)[0]
    check(f"  #{trial + 1}：所有 ω² > 0", np.all(o2_p > 0))
    samples = [(x @ B_p @ x) / (x @ A_p @ x) for x in rng.standard_normal((300, 4))]
    check(f"  #{trial + 1}：ω²_min ≤ 任何 Rayleigh 商 ≤ ω²_max",
          o2_p.min() <= min(samples) + 1e-9 and max(samples) <= o2_p.max() + 1e-9)

# 練習 4
print("\n練習 4：N = 50 的彈簧鏈")
N4 = 50
B4 = second_difference_matrix(N4)
j4 = np.arange(1, N4 + 1)
true1 = np.linalg.eigvalsh(B4)[0]
for name, x in [("常數", np.ones(N4)),
                ("拋物線", (j4 * (N4 + 1 - j4)).astype(float)),
                ("正弦（正解）", np.sin(j4 * np.pi / (N4 + 1)))]:
    val = (x @ B4 @ x) / (x @ x)
    print(f"  {name:<14} λ = {val:.8f}，相對誤差 = {(val - true1) / true1:>8.3%}")
    check(f"  {name} 是上界", val >= true1 - 1e-12)
print(f"  真值 ω₁² = {true1:.8f}（≈ (π/(N+1))² = "
      f"{(np.pi / (N4 + 1)) ** 2:.8f}）")
check("  ω₁² ≈ (π/(N+1))²（連續極限）", true1, (np.pi / (N4 + 1)) ** 2, tol=1e-4)

## 本章重點回顧

* 小振盪的動能與位能都是二次式 ⇒ $A\ddot{\mathbf q}+B\mathbf q=\mathbf 0$ ⇒
  **廣義特徵值問題** $B\mathbf x=\omega^2A\mathbf x$。
* 用 Cholesky $A=R^{\mathsf T}R$ 轉成對稱標準問題，保證 $\omega^2$ 為實數；
  特徵向量在 **$A$-內積**下正交（不是普通內積！），且同時對角化 $A$ 與 $B$。
* **法座標**讓 $N$ 個耦合振盪子變成 $N$ 個獨立單擺；任意運動 = 法模態的疊加。
* CO₂ 模型：$\omega=0$ 的零模態 = $B$ 的零空間 = 整體平移
  （就是第 2 章那個奇異的「自由–自由」矩陣）；振動模態保持質心不動。
* 彈簧鏈的 $B=kK$ ⇒ 法模態就是**離散正弦波**，
  $\omega_n^2=\frac{4k}{m}\sin^2\frac{n\pi}{2(N+1)}$；
  連續極限給出弦的泛音列 $\omega_n\propto n$。
* 系統有對稱性 $\Rightarrow$ 對稱操作與 $A,B$ 可交換 $\Rightarrow$
  模態可依對稱／反對稱分類，大矩陣**分塊對角化**（第 23 章群論的伏筆）。
* **Rayleigh–Ritz**：任何試驗向量都給出 $\omega_1^2$ 的上界；
  因為 Rayleigh 商在特徵向量處駐定，試驗向量的一階誤差只造成二階的頻率誤差。
  子空間 Ritz 法就是有限元素法的核心。

下一章：向量代數、叉積與張量 ── 座標變換下「什麼東西不變」。